# TryOn Studio: resumable CatVTON preparation

Select a T4 GPU and add GITHUB_TOKEN with Contents read access to this repository to Colab Secrets. The setup creates an isolated Python 3.11 environment, while the notebook kernel stays unchanged. It prints four stages and preserves available caches in My Drive/TryOnStudio.

PREPARE_ONLY = True is the default: prepare and check imports without starting a public web worker or tunnel. Free Colab restricts interaction primarily through an external web UI. Serving mode requires a runtime/provider that permits that workflow and a TRYON_WORKER_TOKEN secret.


In [ ]:
# Safe to rerun after setup failure or runtime replacement.
# Keep True for interactive free-Colab preparation.
# Remote serving requires a runtime/provider that permits that workflow.
PREPARE_ONLY = True

from google.colab import drive, userdata
from getpass import getpass
from pathlib import Path
import base64
import os
import subprocess
import sys

def read_secret(name):
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    value = (value or getpass(f"{name} (input hidden): ")).strip()
    if not value:
        raise RuntimeError(f"{name} is empty.")
    return value

def prepare():
    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive/TryOnStudio")
    source = root / "source"
    root.mkdir(parents=True, exist_ok=True)
    token = read_secret("GITHUB_TOKEN")
    encoded = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git_environment = {
        **os.environ,
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraHeader",
        "GIT_CONFIG_VALUE_0": f"Authorization: Basic {encoded}",
        "GIT_TERMINAL_PROMPT": "0",
    }

    def git(arguments):
        try:
            result = subprocess.run(
                ["git", *arguments], env=git_environment,
                capture_output=True, text=True, check=False, timeout=300,
            )
        except subprocess.TimeoutExpired:
            raise RuntimeError("Git took longer than five minutes. Check connectivity and retry.") from None
        if result.returncode:
            message = result.stderr or result.stdout
            for secret in (token, encoded):
                message = message.replace(secret, "[REDACTED]")
            raise RuntimeError("Git failed:\n" + message.strip())

    try:
        if not (source / ".git").exists():
            print("Cloning the private repository...", flush=True)
            git(["clone", "https://github.com/CloudyPatil/ai-virtual-try-on-extension.git", str(source)])
        else:
            print("Fetching the latest project fixes...", flush=True)
            git(["-C", str(source), "rev-parse", "--verify", "HEAD"])
            git(["-C", str(source), "fetch", "origin", "main"])
            git(["-C", str(source), "merge", "--ff-only", "origin/main"])
    finally:
        git_environment.pop("GIT_CONFIG_VALUE_0", None)

    environment = {
        **os.environ,
        "HF_HOME": str(root / "huggingface"),
        "HF_HUB_CACHE": str(root / "huggingface/hub"),
        "PYTHONUNBUFFERED": "1",
    }
    command = [
        sys.executable, "-u",
        str(source / "services/inference-worker/colab_bootstrap.py"),
        "--project-root", str(source),
        "--persistent-root", str(root),
    ]
    if PREPARE_ONLY:
        command.append("--prepare-only")
    else:
        worker_token = read_secret("TRYON_WORKER_TOKEN")
        if len(worker_token) < 24:
            raise RuntimeError("TRYON_WORKER_TOKEN must contain at least 24 characters.")
        environment["TRYON_WORKER_TOKEN"] = worker_token
    result = subprocess.run(command, env=environment, check=False)
    if result.returncode:
        raise RuntimeError("Setup failed. Inspect the last output lines and run the diagnostics cell.")

prepare()


## Completion and recovery

Wait for [4/4] Environment preparation complete. This checks the environment and imports; checkpoints and real inference are not yet tested. Rerun setup after reconnecting: it fetches project fixes and reuses installed packages in the same VM. A new VM recreates the environment from available Drive caches. Do not delete the Drive cache.


In [ ]:
# Diagnostics after a failure; fallback to the previous saved setup log.
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
local = Path("/content/tryon-studio")
setup = local / "setup.log"
if not setup.exists():
    setup = Path("/content/drive/MyDrive/TryOnStudio/setup.log")
print("\n===== setup.log =====")
print(setup.read_text(errors="replace")[-12000:] if setup.exists() else "No setup log found.")
for name in ("worker.log", "tunnel.log"):
    path = local / "logs" / name
    print(f"\n===== {name} =====")
    print(path.read_text(errors="replace")[-6000:] if path.exists() else "Not started in this VM.")
